# Look-Away Tabular Transformer

This notebook trains a small Transformer on the existing window-level visual features. Each numerical feature is represented as a token, and a `[CLS]` token is used for three-class prediction.

> **Important limitation:** the current dataset contains one aggregated row per 3-second window, not a sequence of frame-level observations. This is therefore a **tabular Transformer**, not a temporal video Transformer. It can provide a controlled model comparison, but it cannot recover event timing that is absent from the CSV.

The workflow avoids common leakage problems:

- videos, rather than individual windows, are separated across train/validation/test sets;
- preprocessing is fitted on training data only;
- validation selects the training duration through early stopping;
- the final model is refitted on train + validation data for the selected number of epochs;
- the untouched held-out videos are evaluated once at the end;
- a Random Forest is evaluated on the identical held-out split for comparison.


## 1. Setup

From the project root, install the training dependencies before running this notebook:

```bash
source backend/venv311/bin/activate
python -m pip install -r backend/requirements-training.txt
```

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42
MAX_EPOCHS = 150
EARLY_STOPPING_PATIENCE = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = ["no_looking_away", "looking_away", "frequent_looking_away"]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "training" / "window_training_dataset.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "look_away_transformer.pt"
PREPROCESSOR_PATH = PROJECT_ROOT / "models" / "look_away_transformer_preprocessor.joblib"

def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Dataset: {DATA_PATH}")

## 2. Load Data and Select Non-Label Features

Only automatic visual features are included. Manual labels, metadata, and derived label columns are excluded to prevent target leakage.

In [ ]:
df = pd.read_csv(DATA_PATH)
TARGET = "look_away_level"

LOOK_AWAY_FEATURES = [
    "window_duration",
    "window_frames",
    "eye_contact_ratio",
    "face_visibility_ratio",
    "gaze_center_ratio",
    "valid_gaze_frames",
    "gaze_center_frames",
    "looking_away_total_time",
    "gaze_horizontal_ratio_mean",
    "gaze_horizontal_ratio_std",
    "gaze_vertical_ratio_mean",
    "gaze_vertical_ratio_std",
    "primary_gaze_horizontal_ratio",
    "primary_gaze_vertical_ratio",
    "primary_gaze_zone_ratio",
    "secondary_gaze_zone_ratio",
    "gaze_deviation_from_primary_ratio",
    "gaze_deviation_from_primary_mean",
    "head_pitch_offset_mean",
    "head_pitch_offset_std",
    "head_yaw_mean",
    "head_yaw_std",
    "head_roll_mean",
    "head_roll_std",
    "head_deviation_from_primary_ratio",
    "head_deviation_from_primary_mean",
]

feature_cols = [column for column in LOOK_AWAY_FEATURES if column in df.columns]
missing_features = sorted(set(LOOK_AWAY_FEATURES) - set(feature_cols))
if missing_features:
    print(f"Features absent from this dataset and skipped: {missing_features}")

model_df = df.copy()
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df = model_df[model_df[TARGET].isin(CLASS_LABELS)].copy()
model_df[TARGET] = model_df[TARGET].astype(int)

X = model_df[feature_cols]
y = model_df[TARGET].to_numpy()
groups = model_df["filename"].to_numpy()

print(f"Rows: {len(model_df)}")
print(f"Videos: {model_df['filename'].nunique()}")
print(f"Features: {len(feature_cols)}")
display(model_df[TARGET].value_counts().sort_index().rename("count"))

## 3. Split by Video

The outer split creates an untouched test set. An inner grouped split creates the validation set used only for early stopping. Windows from one video never appear in more than one subset.

In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(outer_splitter.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx]
y_train_val = y[train_val_idx]
groups_train_val = groups[train_val_idx]

inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)
inner_train_rel, val_rel = next(
    inner_splitter.split(X_train_val, y_train_val, groups=groups_train_val)
)
train_idx = train_val_idx[inner_train_rel]
val_idx = train_val_idx[val_rel]

def split_summary(name, indices):
    labels = pd.Series(y[indices]).value_counts().sort_index()
    return {
        "split": name,
        "rows": len(indices),
        "videos": len(set(groups[indices])),
        **{f"level_{level}": int(labels.get(level, 0)) for level in CLASS_LABELS},
    }

display(pd.DataFrame([
    split_summary("train", train_idx),
    split_summary("validation", val_idx),
    split_summary("test", test_idx),
]))
print("Held-out test videos:")
print(sorted(set(groups[test_idx])))

## 4. Preprocess Training Data

Median imputation and standardisation are fitted only on the inner training split while selecting the epoch count.

In [ ]:
selection_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_train = selection_preprocessor.fit_transform(X.iloc[train_idx]).astype(np.float32)
X_val = selection_preprocessor.transform(X.iloc[val_idx]).astype(np.float32)
y_train = y[train_idx].astype(np.int64)
y_val = y[val_idx].astype(np.int64)

def make_loader(features, labels, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(features),
        torch.from_numpy(labels),
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)

class_counts = np.bincount(y_train, minlength=len(CLASS_LABELS))
class_weights = len(y_train) / (len(CLASS_LABELS) * np.maximum(class_counts, 1))
print("Training class counts:", class_counts)
print("Loss weights:", np.round(class_weights, 3))

## 5. Define a Small Numerical-Feature Transformer

Each scalar feature is projected into a learned token embedding. A Transformer encoder models interactions among the features, and the `[CLS]` representation produces the class logits. The network is intentionally small because the dataset is small.

In [ ]:
class NumericalFeatureTransformer(nn.Module):
    def __init__(
        self,
        n_features,
        n_classes=3,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.25,
    ):
        super().__init__()
        self.n_features = n_features
        self.feature_weight = nn.Parameter(torch.empty(n_features, d_model))
        self.feature_bias = nn.Parameter(torch.zeros(n_features, d_model))
        self.feature_identity = nn.Parameter(torch.empty(n_features, d_model))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )
        nn.init.normal_(self.feature_weight, std=0.02)
        nn.init.normal_(self.feature_identity, std=0.02)
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, values):
        feature_tokens = (
            values.unsqueeze(-1) * self.feature_weight.unsqueeze(0)
            + self.feature_bias.unsqueeze(0)
            + self.feature_identity.unsqueeze(0)
        )
        cls = self.cls_token.expand(values.size(0), -1, -1)
        encoded = self.encoder(torch.cat([cls, feature_tokens], dim=1))
        return self.head(encoded[:, 0])

MODEL_CONFIG = {
    "n_features": len(feature_cols),
    "n_classes": len(CLASS_LABELS),
    "d_model": 32,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 64,
    "dropout": 0.25,
}

model = NumericalFeatureTransformer(**MODEL_CONFIG).to(DEVICE)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print(f"Trainable parameters: {trainable_parameters:,}")

## 6. Select the Epoch Count with Early Stopping

Validation macro F1 is used because each look-away level should contribute equally. The test set is not inspected during this stage.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_features)
        loss = criterion(logits, batch_labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(batch_labels)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def predict(model, loader, criterion=None):
    model.eval()
    total_loss = 0.0
    labels_all, predictions_all, probabilities_all = [], [], []
    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        logits = model(batch_features)
        if criterion is not None:
            total_loss += criterion(logits, batch_labels).item() * len(batch_labels)
        probabilities = torch.softmax(logits, dim=1)
        labels_all.append(batch_labels.cpu().numpy())
        predictions_all.append(probabilities.argmax(dim=1).cpu().numpy())
        probabilities_all.append(probabilities.cpu().numpy())
    mean_loss = total_loss / len(loader.dataset) if criterion is not None else np.nan
    return (
        mean_loss,
        np.concatenate(labels_all),
        np.concatenate(predictions_all),
        np.concatenate(probabilities_all),
    )

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

history = []
best_val_f1 = -np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_true, val_pred, _ = predict(model, val_loader, criterion)
    val_f1 = f1_score(val_true, val_pred, average="macro", zero_division=0)
    val_balanced_accuracy = balanced_accuracy_score(val_true, val_pred)
    scheduler.step(val_f1)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_balanced_accuracy": val_balanced_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })

    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | train loss {train_loss:.4f} | "
            f"val loss {val_loss:.4f} | val macro F1 {val_f1:.3f}"
        )
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}")
print(f"Best validation macro F1: {best_val_f1:.3f}")

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].axvline(best_epoch, color="black", linestyle="--", alpha=0.6)
axes[0].set(title="Loss", xlabel="Epoch")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_macro_f1"], label="macro F1")
axes[1].plot(history_df["epoch"], history_df["val_balanced_accuracy"], label="balanced accuracy")
axes[1].axvline(best_epoch, color="black", linestyle="--", alpha=0.6)
axes[1].set(title="Validation Metrics", xlabel="Epoch", ylim=(0, 1))
axes[1].legend()
plt.tight_layout();

## 7. Refit on Train + Validation and Evaluate Once on Test

The selected epoch count is now fixed. A fresh model and preprocessor are trained using all non-test videos, then evaluated once on the held-out videos.

In [ ]:
final_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
X_final_train = final_preprocessor.fit_transform(X.iloc[train_val_idx]).astype(np.float32)
X_test = final_preprocessor.transform(X.iloc[test_idx]).astype(np.float32)
y_final_train = y[train_val_idx].astype(np.int64)
y_test = y[test_idx].astype(np.int64)

final_train_loader = make_loader(X_final_train, y_final_train, shuffle=True)
test_loader = make_loader(X_test, y_test)
final_counts = np.bincount(y_final_train, minlength=len(CLASS_LABELS))
final_weights = len(y_final_train) / (len(CLASS_LABELS) * np.maximum(final_counts, 1))
final_criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(final_weights, dtype=torch.float32, device=DEVICE)
)

set_seed()
final_model = NumericalFeatureTransformer(**MODEL_CONFIG).to(DEVICE)
final_optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
for epoch in range(1, best_epoch + 1):
    train_one_epoch(final_model, final_train_loader, final_optimizer, final_criterion)

_, test_true, test_pred, test_proba = predict(final_model, test_loader)
transformer_macro_f1 = f1_score(test_true, test_pred, average="macro", zero_division=0)
transformer_balanced_accuracy = balanced_accuracy_score(test_true, test_pred)
print(classification_report(
    test_true,
    test_pred,
    labels=CLASS_LABELS,
    target_names=CLASS_NAMES,
    zero_division=0,
))
print(f"Transformer macro F1: {transformer_macro_f1:.3f}")
print(f"Transformer balanced accuracy: {transformer_balanced_accuracy:.3f}")
ConfusionMatrixDisplay.from_predictions(
    test_true,
    test_pred,
    labels=CLASS_LABELS,
    display_labels=["level 0", "level 1", "level 2"],
);

## 8. Compare with Random Forest on the Same Split

A Transformer result is only meaningful when compared against a simpler model trained on exactly the same non-test videos and evaluated on the same held-out videos.

In [ ]:
forest_baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])
forest_baseline.fit(X.iloc[train_val_idx], y[train_val_idx])
forest_pred = forest_baseline.predict(X.iloc[test_idx])

comparison = pd.DataFrame([
    {
        "model": "tabular_transformer",
        "macro_f1": transformer_macro_f1,
        "balanced_accuracy": transformer_balanced_accuracy,
    },
    {
        "model": "random_forest",
        "macro_f1": f1_score(y[test_idx], forest_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y[test_idx], forest_pred),
    },
]).sort_values("macro_f1", ascending=False)
display(comparison.style.format({"macro_f1": "{:.3f}", "balanced_accuracy": "{:.3f}"}))

## 9. Inspect Predictions and Errors

In [ ]:
prediction_df = model_df.iloc[test_idx][["filename", "window_start", "window_end", TARGET]].copy()
prediction_df["predicted_level"] = test_pred
for level in CLASS_LABELS:
    prediction_df[f"probability_level_{level}"] = test_proba[:, level]
prediction_df["correct"] = prediction_df[TARGET] == prediction_df["predicted_level"]
prediction_df["confidence"] = test_proba.max(axis=1)

print("Most confident mistakes:")
display(
    prediction_df[~prediction_df["correct"]]
    .sort_values("confidence", ascending=False)
    .head(20)
)

## 10. Save the Transformer

The model state, architecture settings, feature order, class meanings, held-out video names, and measured test metrics are saved together. The fitted preprocessing pipeline is saved separately.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
artifact = {
    "model_state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in final_model.state_dict().items()
    },
    "model_config": MODEL_CONFIG,
    "feature_cols": feature_cols,
    "class_labels": CLASS_LABELS,
    "class_names": CLASS_NAMES,
    "selected_epoch_count": best_epoch,
    "test_videos": sorted(set(groups[test_idx])),
    "test_macro_f1": transformer_macro_f1,
    "test_balanced_accuracy": transformer_balanced_accuracy,
    "data_path": str(DATA_PATH),
}
torch.save(artifact, MODEL_PATH)
joblib.dump(final_preprocessor, PREPROCESSOR_PATH)
print(f"Saved model: {MODEL_PATH}")
print(f"Saved preprocessor: {PREPROCESSOR_PATH}")

## Interpretation

Do not judge the experiment from training loss. Use held-out macro F1, balanced accuracy, per-level recall, and the confusion matrix. With the current small dataset, results may change noticeably across video splits.

If this tabular Transformer does not outperform Random Forest, that is a useful result rather than a failed experiment: it indicates that model complexity is not compensating for missing temporal features, class imbalance, or label-feature disagreement. A true temporal Transformer should only be attempted after exporting higher-frequency frame-level sequences and adding direct event-duration/count supervision.